# LAX — Run the fine-tuned Streamlit app on Google Colab GPU

Use **Runtime → Change runtime type → T4 GPU** (or another available NVIDIA GPU), then run every cell from top to bottom. This notebook uses the saved `Qwen2.5-1.5B-Instruct` LoRA adapter in Google Drive, 4-bit bitsandbytes loading, BGE-small embeddings, FAISS retrieval, and Streamlit. It must stop early if CUDA is unavailable rather than silently starting CPU mode.


## 1. Mount Drive and confirm the v2 adapter


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
ADAPTER_PATH = Path("/content/drive/MyDrive/psychology-qwen-qlora-v2")
weights_file = ADAPTER_PATH / "adapter_model.safetensors"
config_file = ADAPTER_PATH / "adapter_config.json"
print("Adapter directory:", ADAPTER_PATH)
print("Adapter weights exist:", weights_file.is_file())
print("Adapter config exists:", config_file.is_file())
if not weights_file.is_file() or not config_file.is_file():
    raise FileNotFoundError(
        f"Saved LoRA adapter is incomplete at {ADAPTER_PATH}. "
        "Expected adapter_model.safetensors and adapter_config.json."
    )


## 2. Clone or update the repository


In [ ]:
%cd /content
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/psychology-raft-rag-chatbot")
if not (REPO_DIR / ".git").exists():
    !git clone https://github.com/karthikeyan-crypto/psychology-raft-rag-chatbot.git
else:
    %cd /content/psychology-raft-rag-chatbot
    !git pull origin main

%cd /content/psychology-raft-rag-chatbot
print("App exists:", Path("app.py").exists())
print("Requirements exists:", Path("requirements.txt").exists())


## 3. Create constraints for Colab's existing PyTorch stack


In [ ]:
import importlib.metadata as metadata
import torch
from pathlib import Path

constraints = [f"torch=={torch.__version__.split('+')[0]}"]

try:
    tv_version = metadata.version("torchvision")
    constraints.append(f"torchvision=={tv_version}")
except metadata.PackageNotFoundError:
    pass

Path("/content/lax-colab-constraints.txt").write_text("\n".join(constraints) + "\n")
print("Keeping Colab package versions:")
print("\n".join(constraints))
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. In Colab, select Runtime > Change runtime type > GPU, "
        "save, reconnect, and restart this notebook from the top."
    )


## 4. Install app dependencies without upgrading Colab's Torch stack


In [ ]:
%cd /content/psychology-raft-rag-chatbot
!python -m pip install -q --prefer-binary -c /content/lax-colab-constraints.txt -r requirements.txt


## Optional compatibility check: TorchAO

Some Colab images include an old optional `torchao` package that Transformers may reject at model-load time. LAX uses bitsandbytes for GPU 4-bit loading, not TorchAO; this cell removes TorchAO only when its version is `0.16.0` or older.

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

try:
    installed_torchao = metadata.version("torchao")
except metadata.PackageNotFoundError:
    installed_torchao = None

print("TorchAO before check:", installed_torchao)
if installed_torchao is not None:
    print("Removing optional TorchAO; LAX uses bitsandbytes for 4-bit loading and does not require TorchAO.")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
else:
    print("TorchAO is absent; this is fine for LAX.")


## 5. Verify imports and dependency consistency


In [ ]:
import torch, transformers, peft, bitsandbytes, accelerate, sentence_transformers, faiss, streamlit

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("Accelerate:", accelerate.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)
print("Streamlit:", streamlit.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA was lost after dependency installation. Restart on a GPU runtime before launching LAX.")
print("pip check:")
!python -m pip check


## 6. Configure LAX to use CUDA + the saved LoRA adapter

This explicitly disables CPU fast/Ollama mode for the Streamlit process. The app will load the trained adapter from Drive and use 4-bit bitsandbytes loading on a working GPU runtime.

In [ ]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. Enable a GPU runtime before launching LAX.")
if not (ADAPTER_PATH / "adapter_model.safetensors").is_file():
    raise FileNotFoundError(f"LoRA weights missing: {ADAPTER_PATH / 'adapter_model.safetensors'}")

os.environ["LAX_ADAPTER_PATH"] = str(ADAPTER_PATH)
os.environ["LAX_USE_OLLAMA"] = "0"
os.environ["LAX_CPU_FAST_MODE"] = "0"
os.environ["LAX_LOAD_GENERAL_ON_CPU"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"

print("GPU selected:", torch.cuda.get_device_name(0))
print("LAX_ADAPTER_PATH:", os.environ["LAX_ADAPTER_PATH"])
print("LAX_USE_OLLAMA:", os.environ["LAX_USE_OLLAMA"])
print("LAX_CPU_FAST_MODE:", os.environ["LAX_CPU_FAST_MODE"])
print("Expected inference path: Transformers + saved LoRA adapter + 4-bit bitsandbytes")


## 7. Stop any previous LAX or tunnel process

This prevents a stale CPU-mode Streamlit process from continuing to serve port 8501.

In [ ]:
import subprocess
import time
subprocess.run(["bash", "-lc", "pkill -f '[s]treamlit run app.py' || true"], check=False)
subprocess.run(["bash", "-lc", "pkill -f '[c]loudflared tunnel --url http://127.0.0.1:8501' || true"], check=False)
time.sleep(2)
print("Stopped old LAX/Cloudflare processes, if any.")


## 8. Start Streamlit in GPU mode

The server inherits the GPU and adapter environment configured above.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

%cd /content/psychology-raft-rag-chatbot
log_path = Path("/content/lax.log")
log_handle = log_path.open("w", encoding="utf-8")
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
streamlit_process = subprocess.Popen(
    [sys.executable, "-u", "-m", "streamlit", "run", "app.py",
     "--server.address=0.0.0.0", "--server.port=8501", "--server.headless=true"],
    cwd="/content/psychology-raft-rag-chatbot",
    env=env,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("Streamlit PID:", streamlit_process.pid)
print("GPU:", torch.cuda.get_device_name(0))
print("LAX_USE_OLLAMA:", env.get("LAX_USE_OLLAMA"), "(must be 0)")
print("LAX_CPU_FAST_MODE:", env.get("LAX_CPU_FAST_MODE"), "(must be 0)")
print("Adapter:", env.get("LAX_ADAPTER_PATH"))
print("Check local server status in the next cell.")


## 9. Check the local server

An HTTP health response means Streamlit is listening. The app script may execute when a browser session opens; inspect the log if the page shows an error.

In [ ]:
import time
import urllib.request
from pathlib import Path

time.sleep(8)
try:
    with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=10) as response:
        print("Streamlit health status:", response.status)
except Exception as exc:
    print("Local Streamlit health check failed:", repr(exc))

print("\n--- Recent LAX log ---")
print(Path("/content/lax.log").read_text(errors="replace")[-12000:])
print("\nNext, run the Cloudflare Tunnel cells and open the new URL.")


## 10. Install Cloudflare Tunnel

Run this after Streamlit starts. The downloaded package is saved in Google Drive so it can be reused in later sessions.

In [ ]:
from pathlib import Path
TOOLS_DIR = Path("/content/drive/MyDrive/LAX-tools")
TOOLS_DIR.mkdir(parents=True, exist_ok=True)
DEB_PATH = TOOLS_DIR / "cloudflared-linux-amd64.deb"

if not DEB_PATH.exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /content/cloudflared-linux-amd64.deb
    !cp /content/cloudflared-linux-amd64.deb "$DEB_PATH"

!dpkg -i "$DEB_PATH" >/content/cloudflared-install.log 2>&1 || (cat /content/cloudflared-install.log; exit 1)
!cloudflared --version


## 11. Start the public tunnel

Copy and open the new `trycloudflare.com` URL. Keep this cell running while using LAX.

In [ ]:
!cloudflared tunnel --url http://127.0.0.1:8501 --protocol http2
